# Sesión 3 — gRPC para microservicios de ML (tutorial)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA**

Este notebook es la **parte teórico-práctica guiada** de la Sesión 3. Construimos, de punta a punta y dentro del propio notebook, un servicio de *scoring* por **gRPC**: definimos el contrato en un archivo `.proto`, generamos el código, levantamos el servidor, lo llamamos desde un cliente (llamada simple y *streaming*) y, al final, comparamos la **latencia contra REST**.

> **Entorno.** El curso usa **[uv](https://docs.astral.sh/uv/)**. Instala las dependencias con:
> ```bash
> uv add grpcio grpcio-tools scikit-learn joblib fastapi "uvicorn[standard]" requests
> ```
> y ejecuta este notebook con el kernel de uv (ver *Puesta en marcha* del README raíz). Si prefieres correrlo tal cual, la siguiente celda instala lo necesario.


In [ ]:
# Si ya hiciste `uv add ...`, puedes omitir esta celda.
# (En un entorno uv, el equivalente es: uv add grpcio grpcio-tools scikit-learn joblib fastapi uvicorn requests)
import sys, subprocess
pkgs = ["grpcio", "grpcio-tools", "scikit-learn", "joblib", "fastapi", "uvicorn", "requests"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("Dependencias listas.")

## 1. El problema que resuelve gRPC

REST (Sesión 1) y GraphQL (Sesión 2) son ideales para el **borde externo**: texto (JSON) sobre HTTP/1.1, universal y fácil de operar. Pero cuando **dos servicios internos** de nuestra plataforma deben hablar miles de veces por segundo, con **latencia mínima** y un **contrato estricto**, el texto y HTTP/1.1 se quedan cortos.

**gRPC** es un framework de *Remote Procedure Call* (RPC): hace que llamar a un procedimiento remoto se vea como llamar a una **función local**. Sus tres piezas:

1. **Protocol Buffers** (`.proto`): un lenguaje de definición de interfaz (IDL) para describir mensajes y servicios de forma **tipada**. Se serializa en **binario** (mucho más chico y rápido que JSON).
2. **Código generado**: `protoc` toma el `.proto` y genera el *stub* del cliente y la clase base del servidor.
3. **HTTP/2** como transporte: multiplexado, binario y con **streaming** en ambas direcciones.


## 2. El contrato: `scoring.proto`

Definimos un servicio `Scoring` con dos métodos: uno **unary** (una petición → una respuesta) y uno **server-streaming** (una petición → varias respuestas). Los mensajes `Features` y `Prediction` son el contrato tipado de entrada y salida.

In [ ]:
%%writefile scoring.proto
syntax = "proto3";
package scoring;

message Features {
  repeated double values = 1;   // vector de entrada del modelo
  string model = 2;             // nombre/identificador del modelo
}

message Prediction {
  double score = 1;
  string model_version = 2;
}

service Scoring {
  rpc Predict(Features) returns (Prediction);            // unary: 1 -> 1
  rpc PredictStream(Features) returns (stream Prediction); // server streaming: 1 -> N
}

## 3. Generar los *stubs*

`grpc_tools.protoc` genera dos archivos que **no se editan a mano** (se regeneran si cambia el `.proto`):

- `scoring_pb2.py` → las clases de los **mensajes** (`Features`, `Prediction`).
- `scoring_pb2_grpc.py` → el **stub del cliente** y la **clase base del servidor**.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "grpc_tools.protoc",
                "-I.", "--python_out=.", "--grpc_python_out=.", "scoring.proto"], check=True)
import os
print("Generados:", [f for f in os.listdir('.') if f.startswith('scoring_pb2')])

## 4. Un modelo de ejemplo

Para que el servicio tenga algo que predecir, entrenamos un modelo mínimo de scikit-learn y lo guardamos. En tu Mini-TP usarás **tu propio modelo**.

In [ ]:
from sklearn.linear_model import LogisticRegression
import numpy as np, joblib

rng = np.random.RandomState(0)
X = rng.randn(300, 3)
y = (X.sum(axis=1) > 0).astype(int)
modelo = LogisticRegression().fit(X, y)
joblib.dump(modelo, "model.pkl")
print("Modelo entrenado y guardado en model.pkl")

## 5. El servidor

Puntos clave:

- Heredamos de la clase base generada `ScoringServicer` e implementamos cada método RPC.
- El modelo se carga **una sola vez** al iniciar (nunca dentro del método, sería carísimo por llamada).
- `Predict` devuelve un `Prediction`; `PredictStream` **emite** varios con `yield`.
- Levantamos el servidor con un *pool* de hilos, en el puerto típico de gRPC `50051`.

Para que el notebook sea autocontenido, lo corremos en un **hilo de fondo**.

In [ ]:
import grpc, joblib, threading
from concurrent import futures
import scoring_pb2, scoring_pb2_grpc

modelo = joblib.load("model.pkl")

class ScoringServicer(scoring_pb2_grpc.ScoringServicer):
    def Predict(self, request, context):
        y = modelo.predict([list(request.values)])[0]
        return scoring_pb2.Prediction(score=float(y), model_version="v1")

    def PredictStream(self, request, context):
        # emite una predicción por cada pequeño desplazamiento del vector (ejemplo de streaming)
        for i in range(5):
            desplazado = [v + i * 0.1 for v in request.values]
            y = modelo.predict([desplazado])[0]
            yield scoring_pb2.Prediction(score=float(y), model_version="v1")

servidor = grpc.server(futures.ThreadPoolExecutor(max_workers=4))
scoring_pb2_grpc.add_ScoringServicer_to_server(ScoringServicer(), servidor)
servidor.add_insecure_port("[::]:50051")
servidor.start()
print("Servidor gRPC escuchando en localhost:50051")

## 6. El cliente

El cliente abre un **canal** hacia el servidor y crea un *stub*. A partir de ahí, `stub.Predict(...)` se ve y se comporta como una **función local** tipada. El *streaming* se recorre como un **iterador**.

> En local usamos `insecure_channel`. En producción se usa **TLS** (`grpc.secure_channel`) para autenticar y cifrar.

In [ ]:
import grpc, scoring_pb2, scoring_pb2_grpc

canal = grpc.insecure_channel("localhost:50051")
stub = scoring_pb2_grpc.ScoringStub(canal)

# --- unary: 1 -> 1 ---
peticion = scoring_pb2.Features(values=[0.2, 1.3, 0.7], model="demo")
respuesta = stub.Predict(peticion)
print("Unary  ->", "score:", respuesta.score, "| version:", respuesta.model_version)

# --- server streaming: 1 -> N ---
print("Stream ->", [round(p.score, 3) for p in stub.PredictStream(peticion)])

## 7. Los cuatro tipos de RPC

Ya viste dos. Los cuatro son:

| Tipo | Forma | Ejemplo de ML |
|---|---|---|
| **Unary** | 1 request → 1 response | Puntuar un caso |
| **Server streaming** | 1 request → N responses | Ir emitiendo scores de un lote |
| **Client streaming** | N requests → 1 response | Enviar un flujo y recibir un resumen |
| **Bidirectional** | N ↔ N | Scoring en vivo sobre un flujo continuo |

Todos se declaran igual en el `.proto`, agregando la palabra `stream` del lado del request, del response, o de ambos.

## 8. gRPC vs REST: la latencia

Servimos el **mismo modelo** por REST (FastAPI, como en la Sesión 1) y medimos el tiempo medio de una predicción por ambos caminos. gRPC usa binario sobre HTTP/2 y reutiliza la conexión; REST manda JSON sobre HTTP/1.1. El objetivo no es "ganarle" a REST, sino **ver el orden de magnitud** y entender cuándo conviene cada uno.

In [ ]:
import threading, time, uvicorn, requests
from fastapi import FastAPI
from pydantic import BaseModel

api = FastAPI()
class Entrada(BaseModel):
    values: list[float]

@api.post("/predict")
def predict(e: Entrada):
    y = modelo.predict([e.values])[0]
    return {"score": float(y), "model_version": "v1"}

def _run():
    uvicorn.run(api, host="127.0.0.1", port=8000, log_level="error")

threading.Thread(target=_run, daemon=True).start()
time.sleep(1.5)  # esperar a que levante
print("API REST lista en http://127.0.0.1:8000")

In [ ]:
N = 200
payload = {"values": [0.2, 1.3, 0.7]}

# REST
t = time.perf_counter()
for _ in range(N):
    requests.post("http://127.0.0.1:8000/predict", json=payload)
rest_ms = (time.perf_counter() - t) / N * 1000

# gRPC (reutilizando el stub)
feat = scoring_pb2.Features(values=[0.2, 1.3, 0.7])
t = time.perf_counter()
for _ in range(N):
    stub.Predict(feat)
grpc_ms = (time.perf_counter() - t) / N * 1000

print(f"REST : {rest_ms:.3f} ms/llamada")
print(f"gRPC : {grpc_ms:.3f} ms/llamada")
print(f"gRPC es ~{rest_ms/grpc_ms:.1f}x más rápido en esta prueba local")

> Los números varían según la máquina, pero típicamente gRPC es varias veces más rápido por llamada: paga menos overhead (binario, HTTP/2, conexión reutilizada). Para **tráfico interno intensivo** esa diferencia se multiplica; para el **borde externo**, REST sigue siendo más simple de operar y cachear.

## 9. Cerrar el servidor

Buena práctica: cerrar el canal y detener el servidor al terminar.

In [ ]:
canal.close()
servidor.stop(0)
print("Canal cerrado y servidor detenido.")

## 10. Errores comunes

- **Editar el código generado** (`*_pb2*.py`): se regeneran; toca el `.proto` y vuelve a correr `protoc`.
- **Cambiar el `.proto` y no regenerar**: cliente y servidor dejan de coincidir.
- **Cargar el modelo dentro del método**: cárgalo una vez al iniciar el servidor.
- **Crear un canal por llamada**: es caro; reutiliza el `channel`/`stub`.
- **Sin manejo de errores**: usa `context.set_code(...)` y `context.set_details(...)`.
- **`insecure_channel` en producción**: usa TLS para autenticar y cifrar.

---

### Del tutorial al Mini-TP 3

Este `scoring.proto` es el molde: en el **Mini-TP 3** defines el servicio para **tu** modelo, implementas el servidor, lo llamas desde un cliente (unary + streaming) y comparas la latencia contra tu REST de la Sesión 1. Tienes un starter en **`mini_tp3_actividad.ipynb`**.